# Test Stage 7 — Does it measure a known signal correctly?

**What this stage checks:** the whole instrument, end to end, against a signal whose true
answer you already know.

Every block has passed on its own. This is the check that they add up: that a tone of known
frequency and amplitude, put into IN1, comes out of memory at the right frequency, the right
size, and with a noise floor that behaves as it should.

By this point a failure means the blocks disagree at a join — almost always a scale factor
applied twice, or not at all — rather than a fault inside any one of them.

---

## What you need

| | |
|---|---|
| Stages 1–6 | Passing |
| Signal generator | Sine into **IN1**, amplitude noted, frequency noted |
| Memory region | Found automatically, the same way Stage 6 does |
| After any reboot | `sudo python3 diagnose_hp_ports.py --open-tz` on the board — the memory permission silently reverts at every boot (until the pending rebuild) |

## The predictions being tested

**Frequency.** With the reference tone at `f_lo`, a signal at `f_lo + Δ` must appear at `+Δ`
in the spectrum. The resolution available is `1 / T`, where `T` is the length of the capture:

$$\text{bin spacing} = \frac{1}{T} = \frac{f_{clk}}{N \times R}$$

**Amplitude.** For an input of amplitude *A* in ADC counts, the magnitude in memory should be

$$|z| = A \times 65536$$

built from ×256 in the mixer and ×256 in the decimator, with the correction filter having
unity gain at DC so it does not change the scale.

**Noise floor.** Should fall as √R, above whatever floor the analogue environment imposes —
the same two-component behaviour Stage 4 measured directly.

## Setup

In [ ]:
import os, mmap, struct, time
import numpy as np

FS_CLK = 125e6          # FPGA processing clock [Hz]

# =============================================================================
#  Acquisition system driver  -  register block at 0x40600000 (sys[6])
#
#  Updated 2026-07-30 to the system Stage 6 established and PROVED on hardware:
#  memory port select (CTRL[8], axi0 by default - axi1 silently drops the Q
#  half of every sample on this board), and the write-error flag (STATUS2[3]),
#  without which "accepted" and "actually stored" cannot be told apart.
# =============================================================================
#  CRITICAL RULE: every access is a SINGLE 32-bit read or write. A bulk mmap
#  slice becomes a burst transfer the bus bridge cannot acknowledge; it locks
#  the bus and reboots the board. Do not "optimise" these loops into slices.
#
#  NOTE ON NUMPY: the board's numpy is old. Use np.polyfit rather than
#  np.linalg.lstsq(..., rcond=None), which fails with "a float is required".
# =============================================================================
ACQ_BASE = 0x40600000

R_CTRL       = 0x00   # [0]en [1]soft-rst [2]test [3]tap [4]arm [5]wrap [6]fir-byp [7]dma-clr [8]axi-port(0=axi0)
R_LO_FINC    = 0x04
R_DEC_LOG2   = 0x08
R_SRC_SEL    = 0x0C
R_TAP_SEL    = 0x10   # 0=mixer 1=decimator 2=corrector
R_TAP_SHIFT  = 0x14
R_PEEK_I     = 0x18
R_PEEK_Q     = 0x1C
R_SAMPLE_CNT = 0x20
R_STATUS     = 0x24
R_MEM_START  = 0x28
R_MEM_STOP   = 0x2C
R_MEM_CUR    = 0x30
R_DMA_COUNT  = 0x34
R_STATUS2    = 0x38   # [0]mem overflow [1]filter overrun [2]filter bypassed [3]write error


def s32(v):
    """Reinterpret a 32-bit register value as signed."""
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(object):
    def __init__(self, base=ACQ_BASE):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, 0x1000, offset=base)

    def w(self, off, val): self.m[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):      return struct.unpack('<I', self.m[off:off+4])[0]

    @staticmethod
    def finc(f_lo):
        """f_out = f_clk * finc / 2**32, so finc = f_out/f_clk * 2**32."""
        return int(round(f_lo / FS_CLK * (1 << 32))) & 0xFFFFFFFF

    def setup(self, f_lo, dec_log2, src=0, test=False, tap_en=False,
              tap_sel=0, tap_shift=10, fir_bypass=False,
              arm=False, wrap=True, mem_start=None, mem_stop=None, axi_port=0):
        """Configure and start. ORDER MATTERS: the decimator holds a running
        total that assumes one fixed decimation factor, so always
        disable -> write settings -> reset -> enable."""
        self.w(R_CTRL, 0)                                  # 1. disable
        self.w(R_LO_FINC,   self.finc(f_lo))               # 2. settings
        self.w(R_DEC_LOG2,  dec_log2)
        self.w(R_SRC_SEL,   src)
        self.w(R_TAP_SEL,   tap_sel)
        self.w(R_TAP_SHIFT, tap_shift)
        if mem_start is not None: self.w(R_MEM_START, mem_start)
        if mem_stop  is not None: self.w(R_MEM_STOP,  mem_stop)
        ctrl = (1 |
                (4   if test       else 0) |
                (8   if tap_en     else 0) |
                (16  if arm        else 0) |
                (32  if wrap       else 0) |
                (64  if fir_bypass else 0) |
                (256 if axi_port   else 0))   # CTRL[8]: 0 = axi0 (default - use
                                              # this), 1 = axi1 (drops Q halves
                                              # on this board; diagnostics only)
        self.w(R_CTRL, ctrl | 2 | 128)                     # 3. enable + reset + dma clear
        self.w(R_CTRL, ctrl)                               # 4. running

    def disable(self):  self.w(R_CTRL, 0)
    def peek(self):     return s32(self.r(R_PEEK_I)), s32(self.r(R_PEEK_Q))
    def count(self):    return self.r(R_SAMPLE_CNT)

    def status(self):
        s = self.r(R_STATUS2)
        return dict(mem_overflow=bool(s & 1),
                    filter_overrun=bool(s & 2),
                    filter_bypassed=bool(s & 4),
                    write_error=bool(s & 8))


acq = Acq()
print('acquisition system mapped at 0x%08X' % ACQ_BASE)
st = acq.status()
print('status: memory overflow=%s  filter overrun=%s  filter bypassed=%s  write error=%s'
      % (st['mem_overflow'], st['filter_overrun'], st['filter_bypassed'],
         st['write_error']))


In [ ]:
# ---- memory region and capture: the system Stage 6 established and PROVED --
#
#  REPLACED 2026-07-30. The earlier version predated the Stage 6 debugging and
#  carried two of its documented traps:
#    - it derived the region from a "mem=" kernel argument this board does not
#      use (Stage 6 reads the device tree instead), and
#    - it set MEM_STOP one past the end, when the hardware treats MEM_STOP as
#      the LAST address it may write (set_stop_i in red_pitaya_acq_dma.v).
#  It also predated the port select, the write-error flag, TrustZone and the
#  poison check, so a transport failure could be mistaken for data.
# -----------------------------------------------------------------------------

def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]

def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None

def _dt_dir(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        p = os.path.join(root, path)
        if os.path.isdir(p):
            return p
    return None

# --- pick the region, same ranking as Stage 6 Step 1 -------------------------
#  1. Memory ABOVE what Linux was told exists, first choice: the kernel has
#     never seen it, so it holds no cached second view that could go stale
#     (the 2026-07-29 labuf lesson).
#  2. Otherwise the largest reserved-memory block of at least 1 MB.
candidates = []
blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
if blob and len(blob) >= 8:
    declared_end = _be32(blob, 0) + _be32(blob, 1)
    if declared_end < 512 * 1024**2:
        candidates.append((1 << 40, declared_end, 512 * 1024**2 - declared_end,
                           'above the declared memory - no cached view exists'))
resv = _dt_dir('reserved-memory')
if resv:
    for name in sorted(os.listdir(resv)):
        blob = _dt_read(os.path.join('reserved-memory', name, 'reg'))
        if blob and len(blob) >= 8:
            a, n = _be32(blob, 0), _be32(blob, 1)
            if n >= 1024**2:
                candidates.append((n, a, n, 'reserved-memory: %s' % name))

assert candidates, 'no usable region - run Stage 6 Step 1 for the full picture'
candidates.sort(reverse=True)
_, _a, _n, _why = candidates[0]

#  MEM_STOP IS THE LAST ADDRESS, NOT ONE PAST THE END - hence start + size - 8.
MEM_START = (_a + 0xFFF) & ~0xFFF                 # page aligned
_size     = min(0x00800000, (_a + _n) - MEM_START) & ~7   # 8 MB is ample
MEM_STOP  = MEM_START + _size - 8                 # LAST address, inclusive
print('memory region: %s' % _why)
print('   0x%08X .. 0x%08X inclusive   %.1f MB, room for %d samples'
      % (MEM_START, MEM_STOP + 7, _size / 1024.0**2, _size // 8))

# --- TrustZone: the check that saves the afternoon ---------------------------
#  The boot software leaves memory CLOSED to the FPGA-side writer, and the
#  opening (diagnose_hp_ports.py --open-tz) REVERTS AT EVERY REBOOT. With a
#  bitstream from before 2026-07-30 and the register closed, every capture
#  would come back pure poison. One read here turns that afternoon into one
#  printed instruction. (Bitstreams built after 2026-07-30 present the trusted
#  badge and do not care; the check still costs nothing.)
_fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
_mm = mmap.mmap(_fd, 4096, offset=0xF8000000)
_tz = struct.unpack('<I', _mm[0x430:0x434])[0]
_mm.close(); os.close(_fd)
if not (_tz >> (MEM_START // (64 * 1024**2))) & 1:
    print('')
    print('WARNING: TrustZone is CLOSED for this region (TZ_DDR_RAM = 0x%08X).' % _tz)
    print('   With a bitstream from before 2026-07-30 NOTHING will reach memory.')
    print('   Fix (10 seconds, no rebuild, needed after every reboot):')
    print('       sudo python3 diagnose_hp_ports.py --open-tz')

# --- capture -----------------------------------------------------------------
AXI_PORT = 0        # axi0. Do not change without reading Stage 6: axi1's
                    # processor-side port drops the Q half of every sample on
                    # this board, silently, with no error flagged.
POISON   = 0xDEADBEEF   # never a measurement, never a count: any word still
                        # holding it after a capture was NOT written


def _poison(start, nbytes):
    """Pre-fill the range about to be used, so 'never written' cannot
    masquerade as data (the 2026-07-29 lesson, kept from Stage 6)."""
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, nbytes, offset=start)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close(); os.close(fd)


def capture(f_lo, dec_log2, n_samples, test=False, fir_bypass=False, settle=0.2):
    """Record n_samples into memory and return them as a complex array.

    Same interface as before; underneath it now uses the Stage 6 system:
    axi0, poison pre-fill, and hard checks on the write-error flag and on
    poison in the read-back, so a transport fault raises an error instead of
    returning numbers that look like a measurement."""
    R    = 1 << dec_log2
    need = n_samples * 8
    assert MEM_START + need - 8 <= MEM_STOP, 'capture longer than the region'
    _poison(MEM_START, need)

    acq.setup(f_lo=f_lo, dec_log2=dec_log2, test=test, fir_bypass=fir_bypass,
              arm=True, wrap=False, mem_start=MEM_START, mem_stop=MEM_STOP,
              axi_port=AXI_PORT)
    time.sleep(settle + n_samples * R / FS_CLK)
    acq.w(R_CTRL, acq.r(R_CTRL) & ~16)              # disarm

    st  = acq.status()
    got = min(int(acq.r(R_DMA_COUNT)), n_samples)

    fd   = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    base = (MEM_START // 4096) * 4096
    d    = MEM_START - base
    ln   = ((d + got * 8 + 4096) // 4096) * 4096
    mm   = mmap.mmap(fd, ln, offset=base)
    buf  = mm[d:d + got * 8]                        # bulk read: ordinary memory
    mm.close(); os.close(fd)

    # ---- transport checks: fail loudly, never return plausible garbage ------
    if st['write_error']:
        raise RuntimeError(
            'memory REFUSED writes during the capture. Most likely the board '
            'rebooted and TrustZone closed again: '
            'sudo python3 diagnose_hp_ports.py --open-tz   (see Stage 6)')
    u32 = np.frombuffer(buf, dtype='<u4')
    n_poison = int(np.count_nonzero(u32 == POISON))
    if got and n_poison > max(8, 0.001 * u32.size):     # tolerate chance hits
        odd = int(np.count_nonzero(u32[1::2] == POISON))
        hint = (' All missing words are Q halves - that is the axi1 signature; '
                'check AXI_PORT is 0.') if odd == n_poison else ''
        raise RuntimeError(
            '%d of %d words read back unwritten (poison).%s See Stage 6.'
            % (n_poison, u32.size, hint))
    if st['mem_overflow']:
        print('WARNING: overflow during capture - samples were lost; '
              'treat this record with suspicion')

    raw = np.frombuffer(buf, dtype='<i4').astype(np.float64)
    return raw[0::2] + 1j * raw[1::2], st


---
## Step 1 — Measure the input, so we know what to expect

Read the raw signal on IN1 first, with the acquisition system disabled so the scope sees the
input directly. This gives the amplitude *A* that the prediction is built from.

In [ ]:
SC_BASE, SC_MAP, BUF_A, NMAX = 0x40100000, 0x40000, 0x10000, 1 << 14

class Scope(object):
    def __init__(self):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, SC_MAP, offset=SC_BASE)
    def w(self, o, v): self.m[o:o+4] = struct.pack('<I', v & 0xFFFFFFFF)
    def r(self, o):    return struct.unpack('<I', self.m[o:o+4])[0]
    def _buf(self, off, n):
        ba = bytearray(4 * n)
        for i in range(n):                          # single 32-bit reads: this is
            o = off + 4 * i                         # the register-space bridge
            ba[4*i:4*i+4] = self.m[o:o+4]
        v = np.frombuffer(bytes(ba), dtype='<u4').astype(np.int64) & 0x3FFF
        return np.where(v & 0x2000, v - 0x4000, v).astype(float)
    def grab(self, n=NMAX):
        self.w(0x14, 1); self.w(0x10, n-1)
        self.w(0x00, 2); self.w(0x00, 1); self.w(0x04, 1)
        t0 = time.time()
        while self.r(0x04) != 0:
            if time.time() - t0 > 1.0: break
        time.sleep(0.005)
        a = self._buf(BUF_A, n)
        wp = self.r(0x18) & (NMAX - 1)              # un-wrap the ring buffer
        return np.roll(a, -wp)

sc = Scope()
acq.disable(); time.sleep(0.05)
x = sc.grab()

def est_freq(v, fs):
    v = v - v.mean()
    X = np.abs(np.fft.rfft(v * np.hanning(len(v))))
    k = int(np.argmax(X[1:])) + 1
    if 1 <= k < len(X) - 1:
        l, c, r = (np.log(X[k-1]+1e-9), np.log(X[k]+1e-9), np.log(X[k+1]+1e-9))
        k = k + 0.5*(l-r)/(l-2*c+r)
    return k * fs / len(v)

F_SIG = est_freq(x, FS_CLK)
A_IN  = np.sqrt(2) * np.std(x)          # amplitude from rms: robust to outliers

print('input frequency : %.1f Hz' % F_SIG)
print('input amplitude : %.1f counts (peak)   [full scale 8191]' % A_IN)
print('peak sample     : %.0f counts' % np.abs(x).max())
print()
print('predicted magnitude in memory = A x 65536 = %.3e' % (A_IN * 65536))

# ---------------------------------------------------------------------------
#  REFINE F_SIG WITH THE INSTRUMENT ITSELF   (added 2026-07-30)
#
#  The scope estimate above comes from 16384 samples at 125 MHz - a 131 us
#  glimpse. Frequency resolution is 1/T and nothing else, so its bins are
#  7629 Hz apart, and interpolating between bins is only trustworthy to about
#  a percent of a bin: roughly +-100 Hz. On top of that, the generator and
#  this board run on different crystals, which can disagree by tens of parts
#  per million - another ~100 Hz at a few MHz.
#
#  Step 2 then asks the peak to land within a few hertz. Demanding hertz from
#  a +-100 Hz reference is a trap this project has met before: A TOLERANCE
#  TIGHTER THAN THE MEASUREMENT. The first run failed exactly so: +119.5 Hz,
#  IDENTICAL at OFFSET = 3000 and 5000 - a constant error in the reference,
#  while the instrument's frequency shift was exact both times.
#
#  So refine: park the reference 2 kHz below the coarse estimate and measure
#  where the tone actually lands, using the acquisition system's own 0.13 s
#  capture - 7.5 Hz bins, sub-hertz after interpolation.
#
#  This does NOT make Step 2 circular. The refinement pins F_SIG at ONE
#  reference setting; Step 2 then moves the reference thousands of hertz and
#  demands the peak move by exactly the same amount. That tests what Stage 8
#  will actually rely on: a frequency axis consistent across settings.
#  (Absolute frequency is set by the 125 MHz crystal, which no self-test can
#  check against itself.)
# ---------------------------------------------------------------------------
def peak_freq(zz, fs):
    """Peak of a complex spectrum, interpolated between bins (Hann window)."""
    ww = np.hanning(len(zz))
    m  = np.abs(np.fft.fft(zz * ww))
    k  = int(np.argmax(m))
    l, c, r = (np.log(m[(k - 1) % len(zz)] + 1e-12), np.log(m[k] + 1e-12),
               np.log(m[(k + 1) % len(zz)] + 1e-12))
    den  = l - 2 * c + r
    frac = 0.5 * (l - r) / den if den != 0 else 0.0
    f = ((k + np.clip(frac, -0.5, 0.5)) % len(zz)) * fs / len(zz)
    return f - fs if f > fs / 2 else f


F_COARSE = F_SIG
_park    = 2000.0                       # far enough from DC and its leakage
z_r, _   = capture(f_lo=F_COARSE - _park, dec_log2=10, n_samples=8192)
F_SIG    = (F_COARSE - _park) + peak_freq(z_r, FS_CLK / 1024)

print()
print('refined with the acquisition system (7.5 Hz bins, interpolated):')
print('   coarse (scope, 131 us) : %.1f Hz' % F_COARSE)
print('   refined F_SIG          : %.2f Hz' % F_SIG)
print('   correction             : %+.2f Hz  = %.4f scope bins, well inside the'
      % (F_SIG - F_COARSE, (F_SIG - F_COARSE) / (FS_CLK / 16384)))
print('                            interpolation error a bin that size allows')


---
## Step 2 — Capture, and look at the spectrum

Place the reference tone a known distance below the signal, so the signal appears at a known
positive offset. Then capture, take a Fourier transform, and see where the peak lands.

**Passes if:** the peak is at the offset you asked for, to within a fraction of a bin.

In [ ]:
DEC     = 10                 # R = 1024
R       = 1 << DEC
N       = 16384              # samples to capture
OFFSET  = 3000.0             # put the signal this far above the reference [Hz]

f_out = FS_CLK / R
T     = N / f_out
BIN   = 1.0 / T

print('decimation R      : %d' % R)
print('output rate       : %.2f kHz' % (f_out / 1e3))
print('capture length    : %d samples = %.3f s' % (N, T))
print('bin spacing (1/T) : %.3f Hz' % BIN)
print()

z, st = capture(f_lo=F_SIG - OFFSET, dec_log2=DEC, n_samples=N)
print('captured %d samples, overflow=%s, filter bypassed=%s'
      % (len(z), st['mem_overflow'], st['filter_bypassed']))

w  = np.hanning(len(z))
Z  = np.fft.fft(z * w)
fz = np.fft.fftfreq(len(z), d=1.0 / f_out)
mag = np.abs(Z)

k  = int(np.argmax(mag))
# refine between bins
l, c, r = (np.log(mag[(k-1) % len(z)]+1e-12), np.log(mag[k]+1e-12),
           np.log(mag[(k+1) % len(z)]+1e-12))
den  = l - 2*c + r
frac = 0.5*(l-r)/den if den != 0 else 0.0
f_pk = ((k + np.clip(frac, -0.5, 0.5)) % len(z)) * f_out / len(z)
if f_pk > f_out / 2: f_pk -= f_out

print()
print('expected peak : %+.1f Hz' % OFFSET)
print('measured peak : %+.1f Hz' % f_pk)
print('error         : %+.1f Hz  (%.3f bins)' % (f_pk - OFFSET, (f_pk - OFFSET) / BIN))
print()
ok_f = abs(f_pk - OFFSET) < max(0.5 * BIN, 0.01 * OFFSET)
print('frequency:', 'PASS' if ok_f else 'FAIL')

In [ ]:
import matplotlib.pyplot as plt

# Amplitude calibration: the Hann window and the FFT length both scale the
# peak, so undo both to recover the true amplitude of the tone.
#
# NO FACTOR OF 2 (removed 2026-07-30 - the first run measured a ratio of
# 1.9817 because of it). The familiar doubling belongs to REAL signals, whose
# amplitude is split between matching lines at +f and -f, so a one-sided
# display doubles what it shows. This data is COMPLEX (I and Q together): a
# tone occupies ONE line, nothing is split, and doubling it is exactly the
# "scale applied twice" that this notebook's own header warns about.
scale  = 1.0 / (len(z) * w.mean())
spec   = mag * scale
peak_a = spec.max()

plt.figure(figsize=(11, 4.5))
order = np.argsort(fz)
plt.semilogy(fz[order], spec[order] + 1e-9, lw=0.7)
plt.axvline(OFFSET, color='r', ls='--', lw=1, label='expected %.0f Hz' % OFFSET)
plt.xlabel('frequency relative to the reference tone [Hz]')
plt.ylabel('amplitude')
plt.title('End-to-end spectrum, %d samples at %.1f kHz  (bins %.2f Hz apart)'
          % (len(z), f_out/1e3, BIN))
plt.legend(); plt.grid(True, which='both', alpha=0.35)
plt.tight_layout(); plt.show()

predicted = A_IN * 65536
print('peak amplitude measured  : %.4e' % peak_a)
print('predicted (A x 65536)    : %.4e' % predicted)
print('ratio                    : %.4f' % (peak_a / predicted))
print()
ok_a = 0.85 < (peak_a / predicted) < 1.15
print('amplitude:', 'PASS' if ok_a else 'FAIL')
if not ok_a:
    r = peak_a / predicted
    print('   ratio %.3f - a power of two would point at a misplaced bit window;' % r)
    print('   a factor of 2 exactly often means a scale applied twice or not at all')

---
## Step 3 — Does the noise floor behave?

Increase the averaging and the noise floor should drop, above whatever floor the analogue
environment imposes. Stage 4 measured this directly from the peek registers; here it is
measured from the spectrum, which is how you will actually use the instrument.

**Passes if:** the floor falls as the decimation increases, following √R until it flattens.
Flattening is expected — it is the environment, not the filter. Stage 4 quantified it.

In [ ]:
print('%6s %14s %16s %14s' % ('R', 'bin spacing', 'noise floor', 'improvement'))
print('-' * 56)
floors, Rs = [], []
for L in (8, 9, 10, 11):
    Rv = 1 << L
    zz, _ = capture(f_lo=F_SIG - OFFSET, dec_log2=L, n_samples=8192)
    ww = np.hanning(len(zz))
    ss = np.abs(np.fft.fft(zz * ww)) * 1.0 / (len(zz) * ww.mean())  # complex data: no x2 (see Step 2)
    ff = np.fft.fftfreq(len(zz), d=1.0 / (FS_CLK / Rv))
    # median away from the peak: a robust estimate of the floor
    away = np.abs(ff - OFFSET) > 500
    fl = float(np.median(ss[away]))
    floors.append(fl); Rs.append(Rv)
    print('%6d %11.2f Hz %16.3e %13.2fx'
          % (Rv, (FS_CLK / Rv) / len(zz), fl, floors[0] / fl))

floors = np.array(floors); Rs = np.array(Rs, float)
print()
print('for reference, ideal sqrt(R) improvement from R=%d:' % Rs[0])
print('   ' + '  '.join('%.2fx' % np.sqrt(r / Rs[0]) for r in Rs))
print()
ok_n = floors[-1] < floors[0]
print('noise:', 'PASS' if ok_n else 'FAIL - the floor is not falling at all')

plt.figure(figsize=(7, 4.5))
plt.loglog(Rs, floors, 'o-', label='measured floor')
plt.loglog(Rs, floors[0] * np.sqrt(Rs[0] / Rs), '--', label=r'ideal $1/\sqrt{R}$')
plt.xlabel('decimation factor R'); plt.ylabel('noise floor in the spectrum')
plt.title('Noise floor versus averaging, end to end')
plt.grid(True, which='both', alpha=0.4); plt.legend()
plt.tight_layout(); plt.show()

---
## Step 4 — Confirm the correction is in the path end to end

Capture the same signal near the edge of the window, once with the correction bypassed and
once with it active. Stage 5 measured this through the peek registers; this repeats it through
the full memory path, which is what your data will actually travel through.

**Passes if:** the corrected amplitude is larger, by roughly the droop the filter is designed
to remove at that position.

In [ ]:
EDGE = 0.18 * f_out                      # near the edge of the corrected band

res = {}
for byp in (True, False):
    zz, _ = capture(f_lo=F_SIG - EDGE, dec_log2=DEC, n_samples=8192, fir_bypass=byp)
    ww = np.hanning(len(zz))
    ss = np.abs(np.fft.fft(zz * ww)) * 1.0 / (len(zz) * ww.mean())  # complex data: no x2 (see Step 2)
    res['off' if byp else 'on'] = ss.max()

gain_db = 20 * np.log10(res['on'] / res['off'])
# what the filter should be adding at this position
u = EDGE / f_out
sinc = np.sin(np.pi * u) / (np.pi * u)
predicted_db = -20 * np.log10(abs(sinc) ** 4)

print('signal placed at %.0f Hz from the reference (%.2f x f_out)' % (EDGE, u))
print()
print('  amplitude, correction OFF : %.4e' % res['off'])
print('  amplitude, correction ON  : %.4e' % res['on'])
print('  boost applied             : %+.3f dB' % gain_db)
print('  predicted droop to undo   : %+.3f dB' % predicted_db)
print('  agreement                 : %+.3f dB' % (gain_db - predicted_db))
print()
ok_c = abs(gain_db - predicted_db) < 0.4
print('correction:', 'PASS' if ok_c else 'FAIL')

---
## Stage 7 summary

| Step | What it proves |
|---|---|
| 1 | The input is known: frequency and amplitude measured directly |
| 2 | A tone comes out of memory at the right frequency and the right size |
| 3 | Averaging improves the noise floor end to end, not just in the registers |
| 4 | The droop correction is genuinely in the path the data travels |

**With this passing, the instrument is verified.** Every block has been checked on its own,
and the assembled chain has been checked against a signal whose answer was known in advance.

## What remains

**Stage 8 — the real detection system.** Anything unexpected from here can reasonably be
attributed to the physics rather than the instrument, which is the entire purpose of Stages
1–7.

Two things are still unbuilt, neither blocking:

- **Trigger synchronisation.** The paper starts a capture in step with the feedback
  sequencer, so a measurement can be timed against an excitation pulse. Worth adding when you
  need timed measurement cycles.
- **A control notebook** for routine use, as opposed to these test notebooks.

One number worth carrying forward from Stage 4: the noise floor of about **72 counts
(−41 dBFS)** that averaging cannot remove. If the electron signal turns out to be below that,
the answer is not a longer acquisition — it is finding what is sitting in the band.

In [ ]:
acq.disable()
print('acquisition disabled')
print('final status:', acq.status())